# 01 - Data Understanding (v2)

Purpose: understand the raw v2 dataset before any cleaning or modelling decisions are made. This is the EMBER2018-derived dataset (`data/dataset_pe_v2_full.csv`), built from Elastic's EMBER2018 corpus (real-world PE files submitted to VirusTotal, features extracted by the EMBER project's own pipeline, re-derived here into this project's own 20-feature schema - see `src/constants.py` for the rationale).

Old dataset (`legacy_v1/data/dataset_malwares.csv`, ~19,600 rows) is preserved untouched, not used here.

In [1]:
import pandas as pd
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

df = pd.read_csv("../data/dataset_pe_v2_full.csv")
print("shape:", df.shape)

shape: (800000, 23)


In [2]:
df.head(5)

,Name,SuspiciousImportFunctions,SuspiciousNameSection,SectionsLength,SectionMinEntropy,SectionMaxEntropy,SectionMinRawsize,SectionMaxRawsize,SectionMinVirtualsize,SectionMaxVirtualsize,DirectoryEntryImport,NumberOfImportedFunctions,NumStrings,StringEntropy,AvgStringLength,NumURLs,NumRegistryKeys,NumPaths,NumMZStrings,FileSize,VirtualSize,Malware,OriginalSplit
0,0abb4fda7d5b13801d63bee53e5e256be43e141faa077a...,10,1,5,-0.0000,6.5322,0,27648,6216,172468,9,156,14573,6.5699,5.9721,0,0,3,51,3101705,380928,0,train
1,c9cafff8a596ba8a80bafb4ba8ae6f2ef3329d95b85f15...,20,0,3,3.6912,6.8229,8192,455680,19856,455304,20,631,1803,5.7973,11.1913,0,0,0,3,504320,524288,0,train
2,eac8ddb4970f8af985742973d6f0e06902d42a3684d791...,6,0,4,1.5899,6.8105,4096,81920,1332,81920,6,274,1238,5.7442,12.0202,0,0,0,1,180224,180224,0,train
3,7f513818bcc276c531af2e641c597744da807e21cc1160...,1,3,8,-0.0000,6.5993,0,36864,8,36628,5,8,11166,6.5780,5.8079,0,0,0,41,2377730,77824,0,train
4,ca65e1c387a4cc9e7d8a8ce12bf1bcf9f534c9032b9d95...,12,0,3,4.0991,7.9861,1024,1116672,7180,1118208,6,124,5547,6.5616,6.0434,0,0,0,25,1153808,1167360,0,train


In [3]:
print(df.dtypes)

Name                             str
SuspiciousImportFunctions      int64
SuspiciousNameSection          int64
SectionsLength                 int64
SectionMinEntropy            float64
SectionMaxEntropy            float64
SectionMinRawsize              int64
SectionMaxRawsize              int64
SectionMinVirtualsize          int64
SectionMaxVirtualsize          int64
DirectoryEntryImport           int64
NumberOfImportedFunctions      int64
NumStrings                     int64
StringEntropy                float64
AvgStringLength              float64
NumURLs                        int64
NumRegistryKeys                int64
NumPaths                       int64
NumMZStrings                   int64
FileSize                       int64
VirtualSize                    int64
Malware                        int64
OriginalSplit                    str
dtype: object


In [4]:
feature_cols = [c for c in df.columns if c not in ("Name", "Malware", "OriginalSplit")]
print(f"{len(feature_cols)} feature columns")
print(df[feature_cols].describe().T.to_string())

20 feature columns
                              count          mean           std    min          25%          50%           75%           max
SuspiciousImportFunctions  800000.0  4.032011e+00  5.190041e+00    0.0       0.0000       1.0000  7.000000e+00  8.000000e+01
SuspiciousNameSection      800000.0  1.084240e+00  2.296704e+00    0.0       0.0000       0.0000  2.000000e+00  9.600000e+01
SectionsLength             800000.0  4.984019e+00  2.829057e+00    0.0       3.0000       5.0000  6.000000e+00  1.980000e+02
SectionMinEntropy          800000.0  1.794356e+00  1.793318e+00   -0.0      -0.0000       1.5850  3.320700e+00  7.999900e+00
SectionMaxEntropy          800000.0  6.677967e+00  1.086943e+00   -0.0       6.2010       6.6421  7.668300e+00  8.000000e+00
SectionMinRawsize          800000.0  3.225769e+04  7.190099e+06    0.0     512.0000     512.0000  4.096000e+03  3.331085e+09
SectionMaxRawsize          800000.0  1.065913e+06  2.222620e+07    0.0   46080.0000  200704.0000  8.314880

## Class balance

EMBER2018 is curated to be exactly 50/50 malicious/benign. This is a deliberate research-dataset design choice, not a real-world base rate -- in production, the overwhelming majority of files scanned are benign. Because of this, accuracy alone would be a misleading metric later; evaluation (notebook 06) reports precision/recall/ROC-AUC/confusion matrix instead, which don't assume any particular class ratio, matching standard practice for imbalanced-deployment classifiers trained on balanced data.

In [5]:
counts = df["Malware"].value_counts().sort_index()
pct = df["Malware"].value_counts(normalize=True).sort_index() * 100
print("0 = benign, 1 = malicious")
for label in counts.index:
    print(f"  {label}: {counts[label]:>7} rows ({pct[label]:.2f}%)")

0 = benign, 1 = malicious
  0:  400000 rows (50.00%)
  1:  400000 rows (50.00%)


## Train/test provenance (`OriginalSplit`)

EMBER2018 ships its own official train (6 files, 800,000 rows total: 600,000 labeled + 200,000 unlabeled) and test (1 file, 200,000 rows, fully labeled) split. This project kept that split as the `OriginalSplit` column instead of re-shuffling everything into a fresh random split. Reasons: it matches the published EMBER benchmark exactly (results are directly comparable to the paper's reported numbers), and it avoids any possibility of leakage between train and test that could otherwise be introduced by an in-house re-split. Notebook 02 uses `OriginalSplit == "train"` for training/validation and holds `OriginalSplit == "test"` out completely until the final evaluation in notebook 06.

In [6]:
print(df["OriginalSplit"].value_counts())

OriginalSplit
train    600000
test     200000
Name: count, dtype: int64


## Data quality checks

In [7]:
missing = df.isnull().sum()
print("columns with missing values:", (missing > 0).sum())
print(missing[missing > 0])

columns with missing values: 0
Series([], dtype: int64)


In [8]:
dupes = df["Name"].duplicated().sum()
print(f"duplicate SHA-256 hashes: {dupes}")

duplicate SHA-256 hashes: 0


## Summary

800,000 rows, 20 numeric features + `Name` (SHA-256) + `Malware` (label) + `OriginalSplit` (provenance). Exactly balanced by design. No missing values, no duplicate hashes. This is already far larger and more representative of real-world software than v1's ~19,600-row academic CSV, which was the leading suspected cause of v1's ~90% real-world false-positive rate (see `legacy_v1/README.md`, `project_eatc_status.md` memory). Next: `02_data_preparation.ipynb` covers cleaning, the dataset-size (sample-vs-full) decision, and the train/validation split.